In [3]:
import os
import numpy as np

from evaluate import load

from BudaOCR.Encoder import LabelEncoder, WylieEncoder, StackEncoder
from BudaOCR.Networks import Easter2PlusNetwork
from BudaOCR.Trainer import OCRTrainer

from BudaOCR.Config import CHARSET
from BudaOCR.Utils import (
    build_data_paths,
    create_dir,
    shuffle_data,
    show_image,
    read_stack_file
    )

In [2]:
import sys

print(sys.version)

3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [8]:
# local dir
#dataset_path = "D:/Datasets/KhyentseWangpo"
dataset_path = "D:/Datasets/rNam-rgyal_OCR-Dataset/Dataset"
image_paths, label_paths = build_data_paths(dataset_path, img_file_ext="jpg")
image_paths, label_paths = shuffle_data(image_paths, label_paths)

print(f"Images: {len(image_paths)}, Labels: {len(label_paths)}")

output_dir = os.path.join("Output")
create_dir(output_dir)

Images: 4548, Labels: 4548


In [5]:
image_width = 3200
image_height = 100
wylie_encoder = WylieEncoder(CHARSET)

stack_file = f"tib-stacks.txt"
stacks = read_stack_file(stack_file)
stack_encoder = StackEncoder(stacks)

num_classes = stack_encoder.num_classes()

Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?


In [6]:
network = Easter2PlusNetwork(image_width, image_height, num_classes=num_classes, easter_variant="fixed")
batch_size = 32
workers = 4

Using Easter2 Fixed


In [9]:
ocr_trainer = OCRTrainer(
    network=network,
    label_encoder=stack_encoder,
    workers=workers, 
    image_width=image_width,
    image_height=image_height,
    batch_size=batch_size, 
    output_dir=output_dir, 
    preload_labels=True
    )

ocr_trainer.init(image_paths, label_paths)

Created output directory: Output\2025_12_14_19_44
OCR-Trainer -> Architecture: Easter2Plus
Train Images: 3638, Train Labels: 3638
Validation Images: 455, Validation Images: 455
Test Images: 455, Test Labels: 455
Saved data distribution to: Output\2025_12_14_19_44\data.distribution


100%|██████████| 455/455 [00:01<00:00, 236.75it/s]


Checking DataLoaders..............
Done!


In [10]:
num_epochs = 36
ocr_trainer.train(epochs=num_epochs, check_cer=True, export_onnx=True, silent=False)

Training network....


100%|██████████| 113/113 [00:50<00:00,  2.22it/s]


Epoch 0 done, avg_loss=6.2252, lr=0.000500


100%|██████████| 14/14 [00:04<00:00,  3.35it/s]


Epoch 0 => Val-Loss: 3.378992795944214, Best-loss: None
Saved checkpoint to: Output\2025_12_14_19_44\OCRModel.pth


KeyboardInterrupt: 

In [ ]:
# check dataloader
test_sample = next(iter(ocr_trainer.test_loader))

test_img = test_sample[0][4].numpy()
test_img = np.transpose(test_img, axes=[1, 2, 0])

show_image(test_img)

In [ ]:
# debug start logits for non-zeros
test_sample = next(iter(ocr_trainer.test_loader))
test_logits, gt_labels = ocr_trainer.network.test(test_sample)
pred = np.argmax(test_logits[0], axis=0)
print(pred[:40]) # there should be no leading zeros